# Cross Validation Dataset Generating File

### Load cycling data

In [3]:
import pandas as pd
from pathlib import Path
import h5py

def load_rides_h5(filepath):
    """Read an HDF5 file written by save_rides_h5 into ride_id -> list[DataFrame]."""
    rides = {}
    with h5py.File(filepath, "r") as h5:
        if h5.attrs.get("format") != "ride_segments_v1":
            raise ValueError("Not a supported ride segments HDF5 file")
        for ride_id in sorted(h5.keys()):
            segments = []
            for segment_id in sorted(h5[ride_id].keys()):
                group = h5[ride_id][segment_id]
                data = {}
                for col_id in sorted(group.keys()):
                    col_group = group[col_id]
                    name = col_group.attrs["name"]
                    kind = col_group.attrs["kind"]
                    values = col_group["values"][:]
                    if kind == "datetime":
                        data[name] = pd.to_datetime(values, unit="ns")
                    elif kind == "timedelta":
                        data[name] = pd.to_timedelta(values, unit="ns")
                    elif kind == "string":
                        decoded = [v.decode("utf-8") if isinstance(v, bytes) else str(v) for v in values]
                        missing = col_group["missing"][:]
                        data[name] = [None if is_missing else v for v, is_missing in zip(decoded, missing)]
                    else:
                        dtype = col_group.attrs["dtype"]
                        data[name] = values.astype(dtype)
                segments.append(pd.DataFrame(data))
            rides[ride_id] = segments
    return rides

rides = load_rides_h5("cycling_data.h5")

In [4]:
import random
kfold1 = rides["000"]+rides["001"]+rides["002"]+rides["003"]+rides["015"]
kfold2 = rides["004"]+rides["006"]+rides["007"]
kfold3 = rides["005"]
kfold4 = rides["008"]+rides["009"]+rides["010"]
kfold5 = rides["011"]+rides["012"]+rides["014"]
kfold6 = rides["013"]+rides["016"]+rides["017"]

data = {}
data["fold1_training"] = random.sample(kfold2+kfold3+kfold4+kfold5+kfold6, len(kfold2+kfold3+kfold4+kfold5+kfold6))
data["fold1_testing"] = random.sample(kfold1, len(kfold1))
data["fold2_training"] = random.sample(kfold1+kfold3+kfold4+kfold5+kfold6, len(kfold1+kfold3+kfold4+kfold5+kfold6))
data["fold2_testing"] = random.sample(kfold2, len(kfold2))
data["fold3_training"] = random.sample(kfold1+kfold2+kfold4+kfold5+kfold6, len(kfold1+kfold2+kfold4+kfold5+kfold6))
data["fold3_testing"] = random.sample(kfold3, len(kfold3))
data["fold4_training"] = random.sample(kfold1+kfold2+kfold3+kfold5+kfold6, len(kfold1+kfold2+kfold3+kfold5+kfold6))
data["fold4_testing"] = random.sample(kfold4, len(kfold4))
data["fold5_training"] = random.sample(kfold1+kfold2+kfold3+kfold4+kfold6, len(kfold1+kfold2+kfold3+kfold4+kfold6))
data["fold5_testing"] = random.sample(kfold5, len(kfold5))
data["fold6_training"] = random.sample(kfold1+kfold2+kfold3+kfold4+kfold5, len(kfold1+kfold2+kfold3+kfold4+kfold5))
data["fold6_testing"] = random.sample(kfold6, len(kfold6))

def save_kfolds_h5(data, filepath):
    with h5py.File(filepath, "w") as f:
        for fold_name, segments in data.items():
            group = f.create_group(fold_name)

            for i, segment in enumerate(segments):
                segment_group = group.create_group(f"segment_{i:04d}")

                for col in segment.columns:
                    segment_group.create_dataset(col, data=segment[col].to_numpy())

save_kfolds_h5(data, "kfolds_data.h5")

